In [ ]:
import os
import shutil
from pathlib import Path

print("Attempting to fix tensorflow_datasets import issues...")

# Uninstall conflicting packages
!pip uninstall -y tensorflow tensorflow_datasets protobuf

# Reinstall compatible versions. Let pip resolve dependencies for tensorflow-cpu for broader compatibility.
!pip install tensorflow-cpu tensorflow_datasets

# NOTE: A runtime restart is often required after installing/uninstalling core libraries.
# Please restart the Colab runtime (Runtime > Restart runtime) after this cell finishes.
print("Installation attempt complete. Please restart your Colab runtime now.")

# The rest of the dataset preparation code, to be run AFTER runtime restart
# Define project root and data root as in the notebook
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"

# Check if data already exists to avoid re-downloading
if not DATA_ROOT.exists() or not (DATA_ROOT / "cat").exists() or not (DATA_ROOT / "dog").exists():
    print("Dataset not found. Downloading and preparing using tensorflow_datasets...")

    import tensorflow_datasets as tfds # Re-import after potential re-installation

    # Download and load the cats_vs_dogs dataset using tensorflow_datasets
    # This will cache the dataset in your Colab environment
    (ds_train, ds_test), ds_info = tfds.load(
        'cats_vs_dogs',
        split=['train[:80%]', 'train[80%:]'], # Use train split for both train/test data for simplicity in this lab
        with_info=True,
        as_supervised=True,
        data_dir='./tfds_cache' # Cache location
    )

    # Create target directories
    DATA_ROOT.mkdir(exist_ok=True)
    cat_dir = DATA_ROOT / "cat"
    dog_dir = DATA_ROOT / "dog"
    cat_dir.mkdir(exist_ok=True)
    dog_dir.mkdir(exist_ok=True)

    print("Extracting images to local data/ directories...")
    # Iterate through the dataset and save images to the expected structure
    # This process saves the images from the TFDS cache to our 'data' folder.
    for i, (image_tensor, label_tensor) in enumerate(ds_train.concatenate(ds_test)):
        # Convert TensorFlow tensor to NumPy array
        image_np = image_tensor.numpy()
        label_np = label_tensor.numpy()

        # Convert to PIL Image and save
        # Ensure the image is valid before saving
        try:
            img = Image.fromarray(image_np)
            if label_np == 0: # Cat
                img.save(cat_dir / f"cat_{i:04d}.jpg")
            else: # Dog
                img.save(dog_dir / f"dog_{i:04d}.jpg")
        except Exception as e:
            print(f"Skipping image {i} due to error: {e}")

    print("Dataset prepared successfully in", DATA_ROOT)

    # Clean up the tfds_cache to save space, if desired. This is optional.
    # shutil.rmtree('./tfds_cache', ignore_errors=True)

else:
    print("Dataset already exists.")

Attempting to fix tensorflow_datasets import issues...
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: tensorflow-datasets 4.9.10
Uninstalling tensorflow-datasets-4.9.10:
  Successfully uninstalled tensorflow-datasets-4.9.10
Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 13.1 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages tha

Installation attempt complete. Please restart your Colab runtime now.
Dataset not found. Downloading and preparing using tensorflow_datasets...


AttributeError: module 'tensorflow_datasets' has no attribute 'load'

# Lab 1: NumPy for Cat and Dog Faces

In this notebook, you will treat **cat and dog face images** as NumPy arrays and build a small hand-crafted feature matrix.

This version focuses on core NumPy image operations and keeps the workflow concrete:

- load an image into a NumPy array
- crop and flip with slicing
- normalize to `[0, 1]`
- convert RGB to grayscale
- compute summaries with `axis=`
- apply a small filter with a kernel and matrix multiplication
- flatten an image into one vector
- engineer features with `np.concatenate(...)` and `np.apply_along_axis(...)`
- stack features into a feature matrix for later machine learning work

**Dataset assumption**

Use the curated cat-and-dog-faces dataset extracted into:

`data/`

In [ ]:
!pip install lab_utils
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Safe project root (works in scripts + notebooks)
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data"

LABELS = ("cat", "dog")
LABEL_TO_INDEX = {"cat": 0, "dog": 1}

IMAGE_EXTENSIONS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

SEED = 1234

def label_from_path(path: Path) -> str:
    label = path.parent.name
    if label not in LABEL_TO_INDEX:
        raise ValueError(f"Unexpected label folder: {path}")
    return label


def load_preview_image(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"))


def list_image_paths(label: str) -> list[Path]:
    label_dir = DATA_ROOT / label
    paths = []
    for pattern in IMAGE_EXTENSIONS:
        paths.extend(label_dir.glob(pattern))
    return sorted(paths)

def shuffled_paths(paths: list[Path], seed_offset: int = 0) -> list[Path]:
    rng = np.random.default_rng(SEED + seed_offset)
    indices = rng.permutation(len(paths))
    return [paths[int(idx)] for idx in indices]

def sample_paths(paths: list[Path], count: int, seed_offset: int) -> list[Path]:
    ordered = shuffled_paths(paths, seed_offset=seed_offset)
    return ordered[: min(count, len(ordered))]


def sample_per_class(paths: list[Path], n_per_class: int, seed_offset: int = 0) -> list[Path]:
    sampled = []
    for label_index, label in enumerate(LABELS):
        label_paths = [path for path in paths if label_from_path(path) == label]
        sampled.extend(sample_paths(label_paths, n_per_class, seed_offset + 50 * label_index))
    return sampled

def split_train_test(paths: list[Path], train_ratio: float = 0.7, seed_offset: int = 0):
    shuffled = shuffled_paths(paths, seed_offset)
    split_idx = int(len(shuffled) * train_ratio)
    return shuffled[:split_idx], shuffled[split_idx:]


# Check dataset exists
expected = [
    DATA_ROOT / "cat",
    DATA_ROOT / "dog",
]
if not all(path.exists() for path in expected):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_ROOT}. Expected 'cat' and 'dog' folders."
    )


# Load all paths
cat_paths = list_image_paths("cat")
dog_paths = list_image_paths("dog")
cat_dog_paths = cat_paths + dog_paths

# Split per class (7:3)
cat_train, cat_test = split_train_test(cat_paths, 0.7, seed_offset=0)
dog_train, dog_test = split_train_test(dog_paths, 0.7, seed_offset=100)

# Combine
train_paths = cat_train + dog_train
test_paths = cat_test + dog_test

print(f"Using dataset from: {DATA_ROOT}")
print(f"Found {len(cat_paths)} cat images")
print(f"Found {len(dog_paths)} dog images")

if len(cat_paths) == 0 or len(dog_paths) == 0:
    raise ValueError("No images found. Check folder paths or file extensions.")

FileNotFoundError: Dataset not found at /content/data. Expected 'cat' and 'dog' folders.

In [ ]:
def show_image_gallery(images, titles=None, ncols=5, figsize=(15, 5), suptitle=None):
    """
    Displays a gallery of images using matplotlib.

    Args:
        images (list of np.ndarray): A list of images to display. Each image
                                     should be a NumPy array.
        titles (list of str, optional): A list of titles for each image.
        ncols (int): Number of columns in the image gallery grid.
        figsize (tuple): Figure size (width, height) in inches.
        suptitle (str, optional): Main title for the entire figure.
    """
    if not images:
        print("No images to display.")
        return

    nrows = (len(images) + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten() if nrows > 1 or ncols > 1 else [axes]

    for i, img in enumerate(images):
        if i < len(axes):
            ax = axes[i]
            ax.imshow(img)
            ax.axis('off')
            if titles and i < len(titles):
                ax.set_title(titles[i])

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    if suptitle:
        fig.suptitle(suptitle)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
    plt.show()


In [ ]:
# Example usage of the new show_image_gallery function
# We'll use some of the pre-loaded sample images if available

# Assuming `sample_image` and `cropped_image` from earlier cells are defined.
# If not, this cell might error out until those are run.
if 'sample_image' in locals() and 'cropped_image' in locals():
    show_image_gallery(
        [sample_image, cropped_image],
        titles=["Original Sample Image", "Cropped Sample Image"],
        ncols=2,
        figsize=(8, 4),
        suptitle="Demonstration of custom show_image_gallery"
    )
else:
    print("Sample images (sample_image, cropped_image) are not yet defined to demonstrate the gallery.")
    print("Please run the relevant cells to define them first.")

Sample images (sample_image, cropped_image) are not yet defined to demonstrate the gallery.
Please run the relevant cells to define them first.


In [ ]:
import lab_utils
import os

print('--- Inspecting lab_utils installation ---')
!pip show lab_utils

# Get the actual directory of the installed lab_utils package
found_path = lab_utils.__path__[0]

if found_path:
    print(f'\n--- Listing contents of {found_path} ---')
    !ls -F {found_path}
else:
    print('\nCould not determine lab_utils installation path programmatically. Please check the `pip show lab_utils` output manually.')

--- Inspecting lab_utils installation ---
Name: lab-utils
Version: 0.5.12
Summary: A collection of useful Python modules for laboratory use
Home-page: https://gitlab.ethz.ch/exotic-matter/cw-beam/lab-utils
Author: Carlos Vigo
Author-email: <carlosv@phys.ethz.ch>
License: GPLv3
Location: /usr/local/lib/python3.12/dist-packages
Requires: pandas, psycopg2, python-json-logger, slacker-log-handler, zc.lockfile
Required-by: 

--- Listing contents of /usr/local/lib/python3.12/dist-packages/lab_utils ---
conf/		   database.py	__project__.py	socket_comm.py
custom_logging.py  __init__.py	__pycache__/


### Visual Helper: Preview the Faces Dataset

Before starting the TODOs, look at a few cat and dog face images from the student-specific subset.


In [ ]:
preview_paths = sample_per_class(cat_dog_paths, n_per_class=3, seed_offset=10)
preview_images = [load_preview_image(path) for path in preview_paths]
preview_titles = [f"{label_from_path(path)}: {path.name}" for path in preview_paths]
show_image_gallery(
    preview_images,
    titles=preview_titles,
    ncols=3,
    figsize=(10, 6),
    suptitle="Cat and dog face preview",
)
plt.show()


## Question 1: Load one image into a NumPy array

Write a function that:

- opens one file from disk
- converts it to RGB
- returns an `H x W x C` NumPy array

This is the starting point for every later NumPy operation in the lab.


In [ ]:
def load_image_np(path: Path) -> np.ndarray:
    # TODO: open the file, convert it to RGB, and return an H x W x C NumPy array.
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"))

sample_path = cat_paths[0]
sample_image = load_image_np(sample_path)
print("shape:", sample_image.shape)
print("dtype:", sample_image.dtype)
print("min/max:", sample_image.min(), sample_image.max())
show_image_gallery([sample_image], titles=[sample_path.name], ncols=1, figsize=(4, 4))
plt.show()

NameError: name 'cat_paths' is not defined

## Question 2: Crop the image with slicing

Implement a centered square crop. Keep the crop size at `48 x 48` for the rest of the lab so the crop is visible and later operations stay consistent.


In [ ]:
def center_crop(image: np.ndarray, crop_size: int = 48) -> np.ndarray:
    # TODO: compute the center crop indices and return a square crop.
    h, w, _ = image.shape
    start_h = (h - crop_size) // 2
    start_w = (w - crop_size) // 2
    end_h = start_h + crop_size
    end_w = start_w + crop_size
    return image[start_h:end_h, start_w:end_w]


cropped_image = center_crop(sample_image, crop_size=48)
print("cropped shape:", cropped_image.shape)
show_image_gallery(
    [sample_image, cropped_image],
    titles=["Original", "Center crop"],
    ncols=2,
    figsize=(8, 4),
)
plt.show()

NameError: name 'sample_image' is not defined

## Question 3: Flip the crop horizontally

Mirror the cropped image from left to right using slicing only.


In [ ]:
def flip_horizontal(image: np.ndarray) -> np.ndarray:
    # TODO: return a left-right flipped copy using slicing.
    raise NotImplementedError("Flip the image horizontally with slicing.")


flipped_image = flip_horizontal(cropped_image)
show_image_gallery(
    [cropped_image, flipped_image],
    titles=["Cropped", "Flipped"],
    ncols=2,
    figsize=(8, 4),
)
plt.show()


## Question 4: Normalize pixels to `[0, 1]`

Convert the cropped RGB image from unsigned integers into `float32` values in the range `[0, 1]`.


In [ ]:
def normalize_01(image: np.ndarray) -> np.ndarray:
    # TODO: convert the image to float32 and divide by 255.
    raise NotImplementedError("Normalize pixel values to [0, 1].")


def show_histograms(uint8_img, float_img):
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.hist(uint8_img.ravel(), bins=50)
    plt.title("Before (uint8: 0–255)")

    plt.subplot(1, 2, 2)
    plt.hist(float_img.ravel(), bins=50)
    plt.title("After (float: 0–1)")

    plt.tight_layout()
    plt.show()

sample_float = normalize_01(cropped_image)

# 1. Side-by-side image (Both image will look the same)
show_image_gallery(
    [cropped_image, sample_float],
    titles=["uint8 (0–255)", "float (0–1)"],
    ncols=2,
    figsize=(8, 4),
)

# 2. Stats
print("Before:", cropped_image.dtype, cropped_image.min(), cropped_image.max())
print("After :", sample_float.dtype, sample_float.min(), sample_float.max())

# 3. Histogram
show_histograms(cropped_image, sample_float)

plt.show()


## Question 5: Convert RGB to grayscale

Turn the normalized RGB image into a single grayscale array using standard RGB weights

$GREY = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$.


In [ ]:
def rgb_to_gray(image_float: np.ndarray) -> np.ndarray:
    # TODO: convert normalized RGB values to one grayscale image.
    raise NotImplementedError("Convert RGB to grayscale.")


sample_gray = rgb_to_gray(sample_float)
print("gray shape:", sample_gray.shape)
print("gray dtype:", sample_gray.dtype)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample_float)
axes[0].set_title("Normalized RGB")
axes[0].axis("off")
axes[1].imshow(sample_gray, cmap="gray")
axes[1].set_title("Grayscale")
axes[1].axis("off")
fig.tight_layout()
plt.show()


## Question 6: Use `axis=` to summarize channels

Compute one mean value per color channel with `axis=(0, 1)`, then choose the brightest channel with `np.argmax(...)`.


In [ ]:
CHANNEL_NAMES = np.array(["red", "green", "blue"])


def channel_summary(image_float: np.ndarray) -> tuple[np.ndarray, int]:
    # TODO: compute per-channel means with axis=(0, 1) and return the brightest channel index.
    raise NotImplementedError("Summarize the RGB channels with axis=(0, 1).")


sample_channel_means, sample_brightest = channel_summary(sample_float)
print("channel means:", sample_channel_means)
print("brightest channel:", CHANNEL_NAMES[sample_brightest])
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(CHANNEL_NAMES, sample_channel_means, color=["#E74C3C", "#2ECC71", "#3498DB"])
ax.set_title("Average brightness per channel")
ax.set_ylabel("Mean value")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()


## Question 7: Apply a filter with a kernel and matrix multiplication

Implement a tiny 2D convolution on the grayscale image. At each location:

1. take a `3 x 3` patch
2. flatten the patch and kernel
3. multiply them with `@`

Use the Laplacian kernel from the setup cell.


In [ ]:
EDGE_KERNEL = np.array(
    [
        [0, 1, 0],
        [1, -4, 1],
        [0, 1, 0],
    ],
    dtype=np.float32,
)


def convolve2d_matmul(image_gray: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    # TODO: slide the kernel over the image and use @ on flattened patches and the flattened kernel (or sum of elementwise products) to compute the response at each location.
    raise NotImplementedError("Apply a 2D filter with matrix multiplication.")


sample_filtered = convolve2d_matmul(sample_gray, EDGE_KERNEL)
print("filtered shape:", sample_filtered.shape)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample_gray, cmap="gray")
axes[0].set_title("Grayscale")
axes[0].axis("off")
axes[1].imshow(np.abs(sample_filtered), cmap="magma")
axes[1].set_title("Filtered |response|")
axes[1].axis("off")
fig.tight_layout()
plt.show()


## Question 8: Flatten one image into one vector

Take the grayscale crop and turn it into a one-dimensional vector.


In [ ]:
def flatten_image(image: np.ndarray) -> np.ndarray:
    # TODO: return a 1D vector version of the image.
    raise NotImplementedError("Flatten the image into one vector.")


sample_flat = flatten_image(sample_gray)
print("original shape:", sample_gray.shape)
print("flat shape:", sample_flat.shape)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(sample_flat[:256], color="#4C6FFF")
ax.set_title("First 256 grayscale values after flattening")
ax.set_xlabel("Index")
ax.set_ylabel("Value")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Question 9: Engineer a feature vector with `concatenate` and `apply`

Build one hand-crafted feature vector that combines:

- RGB means
- RGB standard deviations
- the brightest channel index
- the mean and standard deviation of the filtered response
- one summary from `np.apply_along_axis(...)`

Use `np.concatenate(...)` to join the pieces.


In [ ]:
FEATURE_NAMES = [
    "mean_r",
    "mean_g",
    "mean_b",
    "std_r",
    "std_g",
    "std_b",
    "brightest_channel",
    "edge_mean",
    "edge_std",
    "row_std_mean",
]


def extract_features(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    cropped = center_crop(image, crop_size=48)
    image_float = normalize_01(cropped)
    gray = rgb_to_gray(image_float)
    channel_means, brightest_channel = channel_summary(image_float)
    channel_stds = image_float.std(axis=(0, 1)).astype(np.float32)
    filtered = convolve2d_matmul(gray, kernel)
    row_std_profile = np.apply_along_axis(np.std, 1, gray)

    # TODO: use np.concatenate to combine the feature pieces into one float32 vector.
    raise NotImplementedError("Build one hand-crafted feature vector.")


sample_features = extract_features(sample_image, EDGE_KERNEL)
print("feature shape:", sample_features.shape)
fig, ax = plot_feature_vector(sample_features, FEATURE_NAMES, title="Sample NumPy feature vector")
plt.show()


## Question 10: Build and inspect a feature matrix

Apply your feature function to the small balanced train/test subsets from the face dataset.

Tasks:

1. build one feature matrix for the train images and one for the test images
2. return the matching integer labels
3. print the resulting shapes
4. compute an overall feature mean with `axis=0`
5. visualize the feature matrix and the average feature vector


In [ ]:
def build_feature_matrix(paths: list[Path], kernel: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # TODO: apply extract_features to every path and return X (features) and y (integer labels).
    raise NotImplementedError("Build the feature matrix for the dataset subset.")


X_train, y_train = build_feature_matrix(train_paths, EDGE_KERNEL)
X_test, y_test = build_feature_matrix(test_paths, EDGE_KERNEL)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("train class counts:", np.bincount(y_train, minlength=2))
print("test class counts:", np.bincount(y_test, minlength=2))

train_feature_mean = X_train.mean(axis=0)
print("overall train feature mean shape:", train_feature_mean.shape)

fig, ax = plt.subplots(figsize=(10, 4))
image = ax.imshow(X_train, aspect="auto", cmap="viridis")
ax.set_title("Train feature matrix")
ax.set_xlabel("Feature index")
ax.set_ylabel("Image index")
ax.set_xticks(range(len(FEATURE_NAMES)))
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha="right")
fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02)
fig.tight_layout()
plt.show()

fig, ax = plot_feature_vector(train_feature_mean, FEATURE_NAMES, title="Average training feature vector")
plt.show()


## Resources